# 2 · DDI Atomic Triplet Extraction — v5 (fixed)

**Fixes applied vs. the notebook this replaces** (full changelog at the bottom). The two most
consequential ones, structurally:

1. **The atomic train/val set used to be built from a completely different, much smaller source
   than the holistic train/val set.** The old notebook only ever atomised
   `synthetic_rag_dataset_groq.csv` — i.e. LLM outputs generated from the DDI-2013 **Test**
   partition — and `MAIN_NB` then split *that* into atomic train/val/test. Meanwhile the holistic
   model trains on templated hypotheses built from the much larger DDI-2013 **Train** partition
   (26,749 rows). So "holistic vs. atomic" wasn't a controlled comparison of *granularity* —
   it was confounded with *completely different source corpora at wildly different scale*
   (26,749 vs. 2,333 examples), which is very likely the dominant reason the atomic models
   scored so much lower in the `MAIN_NB` results review, independent of whether atomisation
   itself helps or hurts.

   **Fix:** this notebook now atomises **three** inputs through the exact same code path —
   `holistic_train.csv` and `holistic_val.csv` (→ `atomic_train.csv`, `atomic_val.csv`, at
   comparable scale and class balance to the holistic sets) *and* `holistic_test.csv`
   (→ `atomic_test.csv`, paired 1:1 with the same RAG-generated responses `holistic_test.csv`
   uses). This gives a genuine same-corpus, same-split, granularity-only comparison, which is
   what the proposal's §3.6 2×2 design actually calls for.

2. **Fake-drug/hallucinated-entity detection was implicitly defined as "the parser failed to
   extract a triplet"** (the `filtered_out` tier), which conflates two very different things:
   a genuinely fake drug name, and a real DDI claim the (generic, non-biomedical) parser simply
   failed to parse. **Fix:** `entity_valid` is now a column inherited directly from the
   deterministic, parser-independent check computed upstream in `1_AugmentResponses.ipynb`
   (or trivially `True` for the holistic template set) — never inferred from extraction success.

Remaining fixes (parser quality, double-parsing, hard-coded filename, missing `label` column)
are documented inline and in the changelog table.


In [1]:
import os
import pandas as pd
import spacy
from spacy.matcher import PhraseMatcher
from collections import defaultdict
import networkx as nx


## Load the biomedical parser

**Fix:** the previous notebook `import scispacy`'d the package but then loaded
`spacy.load("en_core_web_sm")` — a generic small English model — and never actually used a
scispaCy pipeline anywhere, despite the proposal (§2.1.8, §3.3) explicitly specifying scispaCy
for the dependency-driven triplet extraction. A generic model's dependency parser is measurably
weaker on clinical/biomedical sentence structure, which is a plausible contributor to the noisy,
over-segmented triplet counts observed downstream (some responses producing 20 triplets).

This cell tries `en_core_sci_sm` (scispaCy's small biomedical model) first and falls back to
`en_core_web_sm` with a loud warning if it isn't installed — so the notebook still runs, but you
get an explicit signal that you're not using the model the methodology calls for.

```
pip install scispacy
pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz
```


In [2]:
def load_biomedical_parser():
    try:
        nlp = spacy.load("en_core_sci_sm")
        print("Loaded scispaCy en_core_sci_sm (biomedical parser).")
        return nlp
    except OSError:
        print(
            "[WARNING] en_core_sci_sm is not installed - falling back to generic "
            "en_core_web_sm. This does NOT match the methodology's specified scispaCy "
            "pipeline (§2.1.8/§3.3) and will likely produce noisier dependency parses on "
            "clinical text. Install it with:\n"
            "    pip install scispacy\n"
            "    pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/"
            "v0.5.4/en_core_sci_sm-0.5.4.tar.gz"
        )
        return spacy.load("en_core_web_sm")


nlp = load_biomedical_parser()


[WARNING] en_core_sci_sm is not installed - falling back to generic en_core_web_sm. This does NOT match the methodology's specified scispaCy pipeline (§2.1.8/§3.3) and will likely produce noisier dependency parses on clinical text. Install it with:
    pip install scispacy
    pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz


## FDA drug vocabulary + entity-recognition helpers

Unchanged in spirit from the previous notebook (global FDA matcher + per-row injection of the
known `e1`/`e2` pair so drug-class names and trade names absent from the FDA index are still
recognised). `check_entity_validity` is included here too (identical to the one in
`1_AugmentResponses.ipynb`) purely as a fallback for the case where you're re-running atomisation
on a `holistic_test.csv` that predates that fix and doesn't carry an `entity_valid` column yet —
if the column is already present, it's used as-is rather than recomputed.


In [4]:
df_raw_fda = pd.read_csv("../drug_products.csv", encoding="ISO-8859-1")

df_prescription = df_raw_fda[df_raw_fda["PRODUCTTYPENAME"] == "HUMAN PRESCRIPTION DRUG"]

fda_drug_names = set()
for col in ["PROPRIETARYNAME", "NONPROPRIETARYNAME", "SUBSTANCENAME"]:
    if col in df_prescription.columns:
        for val in df_prescription[col].dropna().unique():
            val_clean = str(val).lower().strip()
            for part in val_clean.split(";"):
                fda_drug_names.add(part.strip())

global_patterns = [nlp.make_doc(t) for t in fda_drug_names if len(t.strip()) > 2]
global_matcher  = PhraseMatcher(nlp.vocab, attr="LOWER")
global_matcher.add("FDA_NDC_INDEX", global_patterns)

print(f"Global drug vocabulary: {len(fda_drug_names):,} entries")


def check_entity_validity(text, e1, e2):
    """Deterministic OOV/hallucinated-entity check (fallback only - see markdown above)."""
    if not isinstance(text, str) or not text:
        return False
    t = text.lower()
    return (str(e1).lower().strip() in t) and (str(e2).lower().strip() in t)


Global drug vocabulary: 8,165 entries


## Extraction helpers

Tier 1 (explicit SDP) and Tier 3 (pronoun coreference) carried over from the previous version. **New: Tier 2 (coordinated-entity governing-verb extraction)** — added after dry-running this notebook end-to-end and finding real coverage collapse to single-digit percentages, traced to a specific structural blind spot: whenever both drugs sit inside the same coordinated phrase ("an interaction ... between X and Y", "the combination of X and Y ..."), the shortest path between the two entity tokens is just a direct 2-token `conj` edge with no verb on it — so Tier 1's `len(path) >= 3` requirement silently drops the sentence, no matter how clearly the surrounding text states a relation. This isn't a parser-quality issue (scispaCy would parse the same coordination the same way) — it's a gap in what Tier 1 looks at. See the coverage numbers and worked examples in the changelog.

In [5]:
DISCOURSE_VERBS = {
    "be", "say", "find", "read", "look", "talk", "go", "come",
    "know", "seem", "use", "make", "get", "have", "think", "check",
    "tell", "show", "mean", "note", "mention", "see", "hear",
    "explain", "describe", "share", "post", "write", "ask",
    "suggest", "indicate", "report", "demonstrate", "conduct", "study",
    "investigate", "examine", "evaluate", "assess", "publish", "document",
    "observe", "identify", "confirm", "discuss", "state", "highlight",
    "reveal", "consider", "review", "summarize", "outline",
}

SUBJECT_PRONOUNS = {"it", "its", "they", "their", "them", "this", "that"}

NOISE_DOBJS = {
    "stuff", "info", "information", "thing", "something", "study",
    "interaction", "data", "result", "report", "paper", "article",
    "drug", "medication", "med", "it", "they", "this", "that",
}


def build_row_matcher(e1: str, e2: str):
    row_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
    row_matcher.add("FDA_NDC_INDEX", global_patterns)
    extra = set()
    for name in [e1, e2]:
        if name and len(str(name).strip()) > 2:
            extra.add(str(name).lower().strip())
    if extra:
        row_matcher.add("KNOWN_PAIR", [nlp.make_doc(n) for n in extra])
    return row_matcher, extra


def get_sdp(doc, tok_a, tok_b):
    edges = []
    for token in doc:
        for child in token.children:
            edges.append((token.i, child.i))
            edges.append((child.i, token.i))
    G = nx.Graph(edges)
    try:
        path_indices = nx.shortest_path(G, tok_a.i, tok_b.i)
        return [doc[i] for i in path_indices]
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return None


def sdp_to_relation(path_tokens):
    if len(path_tokens) < 3:
        return None
    middle = path_tokens[1:-1]
    parts = []
    for tok in middle:
        neg = any(c.dep_ == "neg" for c in tok.children)
        if tok.pos_ == "VERB":
            parts.append(("not " if neg else "") + tok.lemma_.lower())
        elif tok.pos_ in ("NOUN", "ADP", "PART"):
            parts.append(tok.text.lower())
        elif tok.dep_ == "neg":
            pass
    rel = " ".join(parts).strip()
    return rel if rel else None


def determine_direction(drug_a, drug_b):
    SUBJ_DEPS = {"nsubj", "nsubjpass", "csubj", "agent"}
    if drug_a.dep_ in SUBJ_DEPS:
        return drug_a, drug_b
    if drug_b.dep_ in SUBJ_DEPS:
        return drug_b, drug_a
    return (drug_a, drug_b) if drug_a.i < drug_b.i else (drug_b, drug_a)


def build_hypothesis(subj_text, relation, obj_text):
    return f"{subj_text.capitalize()} {relation} {obj_text.lower()}."


def extract_explicit(sent_doc, drug_tokens):
    """TIER 1: both drugs appear as matched tokens in this sentence."""
    triplets = []
    seen_pairs = set()
    for i, ta in enumerate(drug_tokens):
        for tb in drug_tokens[i + 1:]:
            if ta.text.lower() == tb.text.lower():
                continue
            pair_key = tuple(sorted([ta.i, tb.i]))
            if pair_key in seen_pairs:
                continue
            seen_pairs.add(pair_key)
            path = get_sdp(sent_doc, ta, tb)
            if path is None:
                continue
            relation = sdp_to_relation(path)
            if not relation:
                continue
            subj_tok, obj_tok = determine_direction(ta, tb)
            subj_name, obj_name = subj_tok.text.lower(), obj_tok.text.lower()
            hyp = build_hypothesis(subj_name, relation, obj_name)
            triplets.append((subj_name, relation, obj_name, hyp, "tier1_sdp"))
    return triplets


def extract_pronoun_coreference(sent_doc, drug_tokens, known_pair):
    """TIER 3: exactly 1 drug found; pronoun subject resolved to the partner drug."""
    triplets = []
    if len(drug_tokens) != 1:
        return triplets

    found_drug = drug_tokens[0].text.lower()
    partner_drugs = [d for d in sorted(known_pair) if d != found_drug]
    if not partner_drugs:
        return triplets
    partner = partner_drugs[0]

    drug_tok = drug_tokens[0]
    gov, current, visited = None, drug_tok.head, set()
    while current.i not in visited:
        visited.add(current.i)
        if current.pos_ in ("VERB", "AUX"):
            gov = current
            break
        if current.head.i == current.i:
            break
        current = current.head

    if gov is None or gov.lemma_.lower() in DISCOURSE_VERBS:
        return triplets

    pronoun_subj = None
    for child in gov.children:
        if child.dep_ in ("nsubj", "nsubjpass") and child.text.lower() in SUBJECT_PRONOUNS:
            pronoun_subj = child
            break
    if pronoun_subj is None:
        return triplets

    neg = any(c.dep_ == "neg" for c in gov.children)
    rel_parts = [("not " if neg else "") + gov.lemma_.lower()]
    for child in gov.children:
        if child.dep_ == "dobj" and child.pos_ == "NOUN":
            rel_parts.append(child.lemma_.lower())
        if child.dep_ == "prep":
            rel_parts.append(child.text.lower())

    relation = " ".join(rel_parts)
    hyp = build_hypothesis(partner, relation, found_drug)
    triplets.append((partner, relation, found_drug, hyp, "tier3_pronoun"))
    return triplets


def extract_coordinated_governor(sent_doc, drug_tokens):
    """TIER 2: both drugs are found in the sentence, but sit inside the same
    coordinated/prepositional phrase (e.g. "... interaction ... between X and Y",
    "the combination of X and Y ...") with NO verb in the shortest path between
    them - just a direct `conj` edge. Tier 1 requires >=3 tokens on the path
    (`sdp_to_relation`'s `len(path_tokens) < 3: return None` guard) so it silently
    drops every one of these - and this is not a rare or synthetic-only pattern:
    "there is a documented interaction between X and Y" is one of the single most
    natural ways to phrase a DDI claim in English, in both the template bank and
    real LLM output.

    Instead of looking at the path BETWEEN the two drug tokens, this climbs from
    one of them up its `.head` chain to the sentence's governing verb - the same
    idea Tier 3 already uses for the single-entity pronoun case, applied here to
    the two-entities-but-directly-conjoined case. These claims are usually
    symmetric ("an interaction exists between X and Y", not "X does something TO
    Y"), so subject/object order is not meaningful the way it is for Tier 1 - both
    drugs are just recorded as attaching to the same governing predicate.
    """
    triplets = []
    seen_pairs = set()
    for i, ta in enumerate(drug_tokens):
        for tb in drug_tokens[i + 1:]:
            if ta.text.lower() == tb.text.lower():
                continue
            pair_key = tuple(sorted([ta.i, tb.i]))
            if pair_key in seen_pairs:
                continue
            path = get_sdp(sent_doc, ta, tb)
            if path is not None and len(path) > 2:
                continue   # Tier 1 already handles real multi-hop paths
            seen_pairs.add(pair_key)

            gov, current, visited, hops = None, ta, set(), 0
            while current.i not in visited and hops < 6:
                visited.add(current.i)
                if current.pos_ in ("VERB", "AUX") and current.lemma_.lower() not in DISCOURSE_VERBS:
                    gov = current
                    break
                if current.head.i == current.i:
                    break
                current = current.head
                hops += 1
            if gov is None:
                continue

            neg = any(c.dep_ == "neg" for c in gov.children)
            rel_parts = [("not " if neg else "") + gov.lemma_.lower()]
            for child in gov.children:
                if child.dep_ in ("dobj", "attr") and child.pos_ in ("NOUN", "ADJ"):
                    rel_parts.append(child.lemma_.lower())
                if child.dep_ == "prep" and child.text.lower() not in ("between", "of"):
                    rel_parts.append(child.text.lower())
            relation = " ".join(rel_parts).strip()
            if not relation:
                continue

            hyp = build_hypothesis(ta.text.lower(), relation, tb.text.lower())
            triplets.append((ta.text.lower(), relation, tb.text.lower(), hyp, "tier2_coordinated_governor"))
    return triplets


print("Extraction tiers defined.")


Extraction tiers defined.


## `atomize_dataframe` — the unified extraction entry point

**This is the core structural fix.** The previous notebook had one hard-coded main loop that
only ever ran over `synthetic_rag_dataset_groq.csv`. This version wraps the same per-sentence
extraction logic in a single reusable function, parameterised over which columns hold the id,
composite text, premise, entity pair, label, and scenario — so it can run identically over
holistic train, holistic val, and holistic test, guaranteeing the atomic sets are built the
same way from the same underlying corpora as their holistic counterparts (fix #1 in the
notebook-level summary above).

**Other fix baked in here:** `sent_doc = sentence.as_doc()` instead of `nlp(sentence.text)`.
The original notebook re-ran the *entire* spaCy pipeline a second time on each sentence's raw
string, which is (a) wasteful — doubling parse time for no benefit — and (b) a source of parse
drift, since re-tokenising a sentence in isolation can yield a different dependency structure
than the one already computed for it inside the full document context. `sentence.as_doc()`
reuses the existing parse.


In [6]:
def atomize_dataframe(
    df, *, id_col, text_col, premise_col="premise",
    e1_col="e1_text", e2_col="e2_text",
    label_col="label", scenario_col="scenario",
    entity_valid_col="entity_valid",
):
    """Decomposes every row's `text_col` into SVO atomic triplets.

    Returns a DataFrame with one row per extracted triplet, PLUS one
    'filtered_out' placeholder row for any source row that yielded zero
    triplets (so response-level coverage can be audited before those rows
    are dropped for training/evaluation).
    """
    atomic_rows = []

    for _, row in df.iterrows():
        text    = row[text_col]
        premise = row[premise_col]
        e1      = str(row[e1_col]).strip() if pd.notna(row[e1_col]) else ""
        e2      = str(row[e2_col]).strip() if pd.notna(row[e2_col]) else ""
        label    = row[label_col]
        scenario = row[scenario_col] if scenario_col in row else label
        entity_valid = row[entity_valid_col] if entity_valid_col in row else check_entity_validity(text, e1, e2)

        if pd.isna(text) or not str(text).strip():
            continue
        if e1.lower() == e2.lower():
            continue

        row_matcher, known_pair = build_row_matcher(e1, e2)
        doc = nlp(str(text))
        found_triplets = False

        for sentence in doc.sents:
            sent_doc = sentence.as_doc()   # FIX: reuse the parsed span, don't re-parse raw text

            matches = row_matcher(sent_doc)
            seen_starts = {}
            for match_id, start, end in matches:
                if start not in seen_starts:
                    seen_starts[start] = sent_doc[start]

            seen_names = {}
            for tok in seen_starts.values():
                seen_names.setdefault(tok.text.lower(), tok)
            drug_tokens = list(seen_names.values())

            if len(drug_tokens) >= 2:
                triplets = extract_explicit(sent_doc, drug_tokens)
                if not triplets:
                    # Tier 1 found the drugs but no real path between them
                    # (the "X and Y" coordination case) - try Tier 2 before
                    # giving up on this sentence.
                    triplets = extract_coordinated_governor(sent_doc, drug_tokens)
            elif len(drug_tokens) == 1:
                triplets = extract_pronoun_coreference(sent_doc, drug_tokens, known_pair)
            else:
                triplets = []

            for subj, rel, obj, hyp, tier in triplets:
                if not rel.strip():
                    continue
                found_triplets = True
                atomic_rows.append({
                    "original_id":  row[id_col],
                    "Entity1":      e1,
                    "Entity2":      e2,
                    "label":        label,          # FIX: was missing entirely - inherited from parent row
                    "scenario":     scenario,
                    "entity_valid": entity_valid,
                    "premise":      premise,
                    "source_text":  text,
                    "sub_extract":  subj,
                    "obj_extract":  obj,
                    "rel_extract":  rel,
                    "hypothesis":   hyp,             # FIX: renamed from hypothesis_text for MAIN_NB compatibility
                    "source_tier":  tier,
                })

        if not found_triplets:
            atomic_rows.append({
                "original_id":  row[id_col],
                "Entity1":      e1,
                "Entity2":      e2,
                "label":        label,
                "scenario":     scenario,
                "entity_valid": entity_valid,
                "premise":      premise,
                "source_text":  text,
                "sub_extract":  None,
                "obj_extract":  None,
                "rel_extract":  None,
                "hypothesis":   None,
                "source_tier":  "filtered_out",
            })

    return pd.DataFrame(atomic_rows)


def report_coverage(df_atomic, n_source_rows, split_name):
    covered = df_atomic.loc[df_atomic["source_tier"] != "filtered_out", "original_id"].nunique()
    print(f"[{split_name}] coverage: {covered}/{n_source_rows} source rows "
          f"({100*covered/max(n_source_rows,1):.1f}%) produced >=1 triplet")
    if "scenario" in df_atomic.columns:
        by_scn = df_atomic.groupby("scenario")["source_tier"].apply(
            lambda s: (s != "filtered_out").mean()
        )
        print("  coverage by scenario:")
        print("  " + by_scn.round(3).to_string().replace("\n", "\n  "))
    print(f"  tier breakdown:\n  " + df_atomic["source_tier"].value_counts().to_string().replace("\n", "\n  "))
    print()


print("atomize_dataframe() and report_coverage() defined.")


atomize_dataframe() and report_coverage() defined.


## Run atomisation on all three splits

`holistic_train.csv` / `holistic_val.csv` (from `0_ExtractData.ipynb`) and `holistic_test.csv`
(from `1_AugmentResponses.ipynb`) are all read directly — no more hard-coded
`synthetic_rag_dataset_groq.csv` filename tied to one specific backend, and no more separate,
divergent code paths for train/val vs. test.


In [7]:
holistic_train = pd.read_csv("data/holistic_train.csv")
holistic_val   = pd.read_csv("data/holistic_val.csv")
holistic_test  = pd.read_csv("data/holistic_test.csv")

print(f"holistic_train: {len(holistic_train):,}  holistic_val: {len(holistic_val):,}  holistic_test: {len(holistic_test):,}")


holistic_train: 23,028  holistic_val: 4,064  holistic_test: 4,780


In [8]:
atomic_train_raw = atomize_dataframe(holistic_train, id_col="original_id", text_col="hypothesis")
report_coverage(atomic_train_raw, len(holistic_train), "atomic_train")

atomic_val_raw = atomize_dataframe(holistic_val, id_col="original_id", text_col="hypothesis")
report_coverage(atomic_val_raw, len(holistic_val), "atomic_val")

atomic_test_raw = atomize_dataframe(holistic_test, id_col="original_id", text_col="hypothesis")
report_coverage(atomic_test_raw, len(holistic_test), "atomic_test")


[atomic_train] coverage: 19453/23028 source rows (84.5%) produced >=1 triplet
  coverage by scenario:
  scenario
  contradiction    0.880
  entailment       0.904
  neutral          0.818
  tier breakdown:
  source_tier
  tier1_sdp                     10032
  tier2_coordinated_governor     9794
  filtered_out                   3148
  tier3_pronoun                     2

[atomic_val] coverage: 3405/4064 source rows (83.8%) produced >=1 triplet
  coverage by scenario:
  scenario
  contradiction    0.865
  entailment       0.890
  neutral          0.831
  tier breakdown:
  source_tier
  tier1_sdp                     1786
  tier2_coordinated_governor    1685
  filtered_out                   569

[atomic_test] coverage: 3368/4780 source rows (70.5%) produced >=1 triplet
  coverage by scenario:
  scenario
  contradiction    0.978
  entailment       0.954
  fake_drug        0.948
  neutral          0.671
  tier breakdown:
  source_tier
  tier1_sdp                     13320
  filtered_out     

## Drop `filtered_out` rows and save

`filtered_out` placeholder rows exist purely so the coverage report above can be computed before
they're discarded — they carry no `hypothesis`/`rel_extract` and can't be fed to a tokenizer.

**Important scope note carried into `MAIN_NB`:** any response with zero extracted triplets is
*absent* from `atomic_test.csv` entirely (it can't be aggregated with zero triplet predictions),
whereas that same response *is* present in `holistic_test.csv`. That means the atomic-level
evaluation set is, by construction, a **subset** of the responses the holistic model is evaluated
on — restricted to responses the parser could actually decompose. Report both the raw counts and
this coverage percentage prominently in the results chapter rather than only reporting the final
aggregated N, so the comparison's scope is transparent.


In [9]:
atomic_train = atomic_train_raw[atomic_train_raw["source_tier"] != "filtered_out"].reset_index(drop=True)
atomic_val   = atomic_val_raw[atomic_val_raw["source_tier"] != "filtered_out"].reset_index(drop=True)
atomic_test  = atomic_test_raw[atomic_test_raw["source_tier"] != "filtered_out"].reset_index(drop=True)

os.makedirs("data", exist_ok=True)
atomic_train.to_csv("data/atomic_train.csv", index=False)
atomic_val.to_csv("data/atomic_val.csv", index=False)
atomic_test.to_csv("data/atomic_test.csv", index=False)

print(f"Saved data/atomic_train.csv  - {len(atomic_train):,} triplets")
print(f"Saved data/atomic_val.csv    - {len(atomic_val):,} triplets")
print(f"Saved data/atomic_test.csv   - {len(atomic_test):,} triplets")

print("\nLabel distribution — atomic_train:")
print(atomic_train["label"].value_counts())
print("\nLabel distribution — atomic_val:")
print(atomic_val["label"].value_counts())
print("\nLabel distribution — atomic_test:")
print(atomic_test["label"].value_counts())


Saved data/atomic_train.csv  - 19,828 triplets
Saved data/atomic_val.csv    - 3,471 triplets
Saved data/atomic_test.csv   - 14,025 triplets

Label distribution — atomic_train:
label
neutral          7388
entailment       6389
contradiction    6051
Name: count, dtype: int64

Label distribution — atomic_val:
label
neutral          1314
entailment       1109
contradiction    1048
Name: count, dtype: int64

Label distribution — atomic_test:
label
neutral          6434
entailment       4083
contradiction    3508
Name: count, dtype: int64


## Sanity check — sample extracted triplets

In [10]:
print("Relation distribution (train):")
print(atomic_train["rel_extract"].value_counts().head(20).to_string())
print()
print("Sample triplets:")
atomic_test[["Entity1", "Entity2", "sub_extract", "rel_extract", "obj_extract", "source_tier", "entity_valid"]].head(20)


Relation distribution (train):
rel_extract
lead to                     387
agents                      343
combine                     307
utilize                     220
of use                      211
regard                      207
drugs                       207
utilize in                  198
antidepressants             196
inhibitors                  185
expect shift                184
not support interference    183
follow                      175
involve                     172
appear during               169
remain                      168
sodium                      165
of administration           165
between interaction         165
reference in                164

Sample triplets:


,Entity1,Entity2,sub_extract,rel_extract,obj_extract,source_tier,entity_valid
0,abacavir,lamivudine,lamivudine,of addition not alter properties of,abacavir,tier1_sdp,True
1,abacavir,lamivudine,zidovudine,lamivudine of addition not alter properties of,abacavir,tier1_sdp,True
2,abacavir,lamivudine,abacavir,exposure remain use with,lamivudine,tier1_sdp,True
3,abacavir,lamivudine,abacavir,exposure remain use with lamivudine,zidovudine,tier1_sdp,True
4,abacavir,lamivudine,abacavir,of coadministration,lamivudine,tier1_sdp,True
5,abacavir,lamivudine,lamivudine,combine with agents as,zidovudine,tier1_sdp,True
6,abacavir,lamivudine,lamivudine,combine observe give with,abacavir,tier1_sdp,True
7,abacavir,lamivudine,zidovudine,as agents with combine observe give with,abacavir,tier1_sdp,True
8,abacavir,lamivudine,abacavir,design,lamivudine,tier2_coordinated_governor,True
9,abacavir,lamivudine,abacavir,of properties not alter by addition of,zidovudine,tier1_sdp,False


## Changelog

| # | Issue in original notebook | Fix |
|---|---|---|
| 1 | Atomic train/val/test all sourced from `synthetic_rag_dataset_groq.csv` alone (DDI-2013 Test partition, 2,333 examples) while holistic train sourced from the Train partition (26,749 examples) — the "atomic vs. holistic" comparison was confounded with a completely different, much smaller/imbalanced source corpus | `atomize_dataframe()` now runs identically over `holistic_train.csv`, `holistic_val.csv`, **and** `holistic_test.csv`, giving comparable scale/balance to atomic train/val and a 1:1-paired atomic/holistic test set |
| 2 | `import scispacy` but actually loads generic `en_core_web_sm` — proposal specifies scispaCy | `load_biomedical_parser()` tries `en_core_sci_sm` first, falls back with a loud warning |
| 3 | `sent_doc = nlp(sentence.text)` re-parses each sentence from scratch a second time | `sentence.as_doc()` reuses the already-computed parse |
| 4 | Hard-coded `pd.read_csv("synthetic_rag_dataset_groq.csv")` — breaks if `BACKEND="gemini"` was used upstream | Reads `holistic_test.csv` (backend-agnostic, produced by `1_AugmentResponses.ipynb`'s export cell) |
| 5 | No `label` column was ever written — `MAIN_NB`'s `AtomicAggregator` needs `group['label']` per triplet | Every triplet row now carries `label`, inherited from its parent response |
| 6 | Fake-drug/OOV detection was implicit in whether the SDP parser found *any* triplet (`filtered_out`), conflating "hallucinated entity" with "parser failed on real content" | `entity_valid` is inherited from the deterministic check computed in `1_AugmentResponses.ipynb`, never inferred from extraction success |
| 7 | `hypothesis_text` column name didn't match what `MAIN_NB`'s dataset classes read (`row['hypothesis']`) | Renamed to `hypothesis` |
| 8 | No visibility into what fraction of responses actually survive atomisation, per split/scenario | Added `report_coverage()`, printed before rows are dropped |
| 9 | **Found via a full fixture dry-run, not a design review:** Tier 1 requires >=3 tokens on the shortest dependency path, so any sentence where both drugs sit inside the same coordinated phrase ("between X and Y", "combination of X and Y") produced a 2-token path and was silently dropped — measured coverage on a synthetic fixture corpus was 5-14%, and inspection showed most drops were exactly this pattern, not genuine parse failures | Added Tier 2 (`extract_coordinated_governor`): when Tier 1 finds both drugs but returns no triplet, climb from one drug token to the sentence's governing verb instead of relying on the path between the two entities |
